# block34: is $+0.0109$ the backbone or the threshold? (2026-08-31)

**Press Run All.** The first useful result arrives in about half an hour and
prints itself. Everything after that is optional.

### What this asks

`block34` is the sweep's only nominally significant backbone effect:
$+0.0109$ precision over the frozen arm at $t = +2.45$. The head ablation was
checked on 2026-08-30 for a confound that applies to `block34` identically, and
the check changed the answer. The band split's $+0.0096$ at fitted thresholds
fell to $+0.0021$ measured threshold-free, because the arms' LOSO-fitted
thresholds land in different places (0.287 against 0.544) and their per-station
precision and recall differences anti-correlate at $r = -0.59$. Most of the
apparent architectural gain was the operating point sliding along one curve,
which a reader can do for free without rewriting their head.

`block34` has never been checked for it, and the check needs no training: the
2026-08-21 run saved each fold's whole model, trunk included, and they are still
on Drive. Average precision integrates over every threshold, so it cannot be
moved by where a fitted cut happened to land.

### Why the retrain is last

A second unseeded draw of `block34` answers reproducibility instead, and costs
about 6.5 hours on a T4. Two attempts at it both died inside 90 minutes, so it
runs last, after the result that fits in a session. It resumes: finished folds
are in Drive and are skipped, so a dead session costs the fold in flight and
nothing before it. Three folds are already done.

There is no feature-cache step. Both jobs read the image pack directly.


In [ ]:
from google.colab import drive; drive.mount('/content/drive')
!rm -rf /content/repo && git clone -q -b v13-honest-labels https://github.com/Mo119m/primates-sound-detection /content/repo
%cd /content/repo
!git log --oneline -1
U = '/content/drive/MyDrive/primates-sound-detection'
!mkdir -p /content/dataF && cp -n {U}/v13_images.npy {U}/v13_index.csv {U}/manifest.csv /content/dataF/
!ls -la /content/dataF

In [ ]:
# Is the Drive copy the same bytes as the build every number in the paper
# comes from? A row count is not enough -- two builds can share one and differ
# in content -- and these Drive files are a 2026-08-20 upload that nothing has
# re-verified since. The digests are from the local full_2026-08-19 artifacts,
# whose mtimes are all 2026-08-19, i.e. before the upload and unmoved since.
# e79cfaee6813 is the index sha every run.json in the shipped sweep records.
import hashlib
import pandas as pd

EXPECT = {
    'v13_index.csv':  'e79cfaee6813075553f8d4878ea54593b093a114e0f8456efe728e16a1f92ab5',
    'manifest.csv':   'a93f43d4d82115abfa1d6399f0756672b66bbadd69855ddbc402edca45c89dd2',
    'v13_images.npy': 'af9827632060ab8afe5b97f7f0b2034442c1d812c0dcdbaa5c80301238debb81',
}
for name, want in EXPECT.items():
    h = hashlib.sha256()
    with open(f'/content/dataF/{name}', 'rb') as fh:
        for chunk in iter(lambda: fh.read(1 << 24), b''):
            h.update(chunk)
    got = h.hexdigest()
    print(f'{name:16s} {got[:12]}  {"OK" if got == want else "MISMATCH"}')
    assert got == want, (
        f'{name} on Drive is not the build this paper reports. Re-upload from '
        f'data/outputs/v13_runs/full_2026-08-19/ before running anything.')

i = pd.read_csv('/content/dataF/v13_index.csv')
assert len(i) == 22169, 'wrong dataset on Drive'
print(len(i), 'rows;', i.label.value_counts().to_dict())
print('Drive artifacts are byte-identical to the shipped build')

In [ ]:
# The time gate needs its lookup table. Without it a fold trains fine and comes
# out missing the four columns every comparison reads -- which happened on
# 24 August and cost seven folds. Fail here, where it is free.
import os
import pandas as pd
G = '/content/repo/data/outputs/auto_cleanup/review_gate_table.csv'
assert os.path.exists(G), 'no review gate table in the clone -- pull the branch again'
g = pd.read_csv(G)
assert len(g) == 6189, 'wrong gate table'
assert set(g.columns) == {'file', 'timestamp', 'start_s'}, 'unexpected columns'
print(len(g), 'rows,', list(g.columns), '-- time gate has its clock')

---

## 1. The threshold-free comparison (about 30 minutes, trains nothing)


In [ ]:
# Scores block34 and block4 from the weights already in Drive, as average
# precision over the same evaluation rows. Writes arms_prauc.csv into the arm
# directory fold by fold, so an interrupted session keeps everything finished.
!REP_ARM_DIR=/content/drive/MyDrive/primates-sound-detection/unfreeze_2026-08-21 python colab/prauc_arms.py

In [ ]:
# The answer, printed here so a session that dies later still leaves it behind.
#
# frozen's per-station average precision is embedded rather than read: it lives
# in a gitignored file (data/outputs/v13_runs/full_2026-08-19/
# head_ablation_prauc.csv, arm 'freqpos', macro 0.9825) that a fresh clone
# cannot see. scripts/verify_manuscript_numbers.py recomputes it at home.
import numpy as np
import pandas as pd

FROZEN_AP = {
    'IPA1ST': 0.9812, 'IPA2ST': 0.9874, 'IPA4ST': 0.9946, 'IPA6ST': 0.9951,
    'IPA7ST': 0.9363, 'IPA8ST': 0.9946, 'IPA10ST': 1.0000, 'IPA11ST': 0.9017,
    'IPA13ST': 0.9858, 'IPA14ST': 0.9854, 'IPA15ST': 0.9987, 'IPA16ST': 0.9882,
    'IPA17ST': 0.9852, 'IPA18ST': 0.9999, 'IPA19ST': 0.9872, 'IPA20ST': 0.9989,
}
P = '/content/drive/MyDrive/primates-sound-detection/unfreeze_2026-08-21/arms_prauc.csv'
d = pd.read_csv(P)
print(f'{len(d)} arm-station scores\n')
for arm, g in d.groupby('arm'):
    x = np.array([g.set_index('station').ap[s] - FROZEN_AP[s]
                  for s in g.station if s in FROZEN_AP])
    if len(x) < 3:
        print(f'  {arm}: only {len(x)} folds so far, too few to pair')
        continue
    se = x.std(ddof=1) / np.sqrt(len(x))
    print(f'  {arm:8s} vs frozen, threshold-free: {x.mean():+.4f} '
          f'(t {x.mean()/se:+.2f}, better at {int((x>0).sum())}/{len(x)}, '
          f'n={len(x)})')
print('\nRead against the fitted-threshold figures the sweep reports:')
print('  block34 +0.0109 (t +2.45),  block4 +0.0050 (t +1.74)')
print('and against the measured noise floor, three draws of one specification:')
print('  largest pairwise difference 0.0035 in precision.')
print('\nIf most of block34 was the operating point, it shrinks here the way')
print('the band split shrank from +0.0096 to +0.0021.')

---

## 2. Optional: a second draw of block34 (about 6.5 hours, resumable)

Only worth starting if the GPU is free for a long stretch. Three of sixteen
folds are already in Drive and will be skipped. Re-running this cell after a
disconnect picks up where it stopped.


In [ ]:
# REP_ARMS defaults to block34_rep2. The other two replicate arms ran locally
# on 2026-08-30 -- they train a frozen trunk on cached features, need no GPU,
# and the local machine is three times faster at it.
!python colab/run_replicates.py